# Dataset 4 Setup, Loading, and Cleaning

#### Setup and imports
Python Virtual Environment Commands

In [18]:
# using a virtual environment ensures that all the packages you install (pandas, numpy, etc.)
# are isolated from your system Python. this helps your project work consistently whether
# you run it on your own laptop or in another environment. it's completely fine if you don't want
# to set this up, you can still run the notebook without it.

# create a virtual environment named 'venv':
# python3 -m venv venv

# activate the virtual environment (Windows or macOS/Linux):
# venv\Scripts\activate
# source venv/bin/activate

# once in the virtual environment, install the right packages:
# pip install pandas numpy matplotlib seaborn

# run the notebook as usual

# deactivate the virtual environment:
# deactivate

In [19]:
# data handling
import pandas as pd   # primary library for loading, inspecting, cleaning, and manipulating tabular data
import numpy as np    # numerical computing library, useful for calculations and handling missing values

# visualization (optional but helpful during cleaning)
import matplotlib.pyplot as plt  # basic plotting library for creating line charts, histograms, and scatterplots
import seaborn as sns            # higher-level visualization library for statistical plots and easier styling

# file handling / os operations
import os  # provides utilities to navigate directories, check file paths, and manage file locations

# optional: warnings suppression for a cleaner notebook
import warnings
warnings.filterwarnings('ignore')  # suppresses unnecessary warnings to make notebook outputs cleaner

# starter settings
pd.set_option('display.max_columns', None)  # show all columns when printing a dataframe
pd.set_option('display.max_rows', 20)      # limit number of rows displayed to avoid clutter
pd.set_option('display.float_format', '{:.2f}'.format)  # format floating numbers to 2 decimals

sns.set_style('whitegrid')  # makes plots cleaner and more readable

#### Loading Raw Dataset - [NYC Permitted Event Information](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information/tvpp-9vvx/about_data)



In [20]:
# Loads the dataset into a pandas DataFrame
dataset4raw = pd.read_csv('../../data/raw/NYC_Permitted_Event_Information-Historical.csv')  
# Displays the first few rows to verify import happened
dataset4raw.head()  

,Event ID,Event Name,Start Date/Time,End Date/Time,Event Borough,Event Location,Event Street Side,Street Closure Type,Police Precinct
0,741181,Miscellaneous,02/01/2024 01:00:00 AM,02/01/2024 10:00:00 PM,Brooklyn,Fort Hamilton Athletic Field (Ft. Hamilton HS)...,NaN,NaN,"68,"
1,741181,Miscellaneous,02/01/2024 01:00:00 AM,02/01/2024 10:00:00 PM,Brooklyn,Fort Hamilton Athletic Field (Ft. Hamilton HS)...,NaN,NaN,"68,"
2,741181,Miscellaneous,02/01/2024 01:00:00 AM,02/01/2024 10:00:00 PM,Brooklyn,Fort Hamilton Athletic Field (Ft. Hamilton HS)...,NaN,NaN,"68,"
3,741181,Miscellaneous,02/01/2024 01:00:00 AM,02/01/2024 10:00:00 PM,Brooklyn,Fort Hamilton Athletic Field (Ft. Hamilton HS)...,NaN,NaN,"68,"
4,741181,Miscellaneous,02/01/2024 01:00:00 AM,02/01/2024 10:00:00 PM,Brooklyn,Fort Hamilton Athletic Field (Ft. Hamilton HS)...,NaN,NaN,"68,"


#### Inspect Dataset for if it needs to be cleaned or not

In [21]:
# Look at the general information of the datset to understand its setup and structure
dataset4raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 502426 entries, 0 to 502425
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   Event ID             502426 non-null  int64
 1   Event Name           502359 non-null  str  
 2   Start Date/Time      502426 non-null  str  
 3   End Date/Time        502426 non-null  str  
 4   Event Borough        502426 non-null  str  
 5   Event Location       502426 non-null  str  
 6   Event Street Side    29602 non-null   str  
 7   Street Closure Type  30486 non-null   str  
 8   Police Precinct      502426 non-null  str  
dtypes: int64(1), str(8)
memory usage: 34.5 MB


In [22]:
# Total datapoints in the dataset
total_datapoints = len(dataset4raw)

# Checks for the amount of missing values in each column
missingValsCount = dataset4raw.isnull().sum()
print("Missing values data percentage:\n" + str(missingValsCount / total_datapoints * 100), end="")

# Checks for the amount of duplicated rows in the dataset
duplicatedValsCount = (dataset4raw.duplicated().sum())
print("Duplicated rows data percentage:", duplicatedValsCount / total_datapoints * 100)

Missing values data percentage:
Event ID               0.00
Event Name             0.01
Start Date/Time        0.00
End Date/Time          0.00
Event Borough          0.00
Event Location         0.00
Event Street Side     94.11
Street Closure Type   93.93
Police Precinct        0.00
dtype: float64Duplicated rows data percentage: 93.37773124798477


Conclusion Reached:

Any column with "Date/Time" should be converted from string to datetime to allow for easier handling later on. 

There are no duplicate rows but there are null values primarily in "Event Street Side" and "Street Closure Type." An unexpected discovery was that many of these data entries do not indicate a street closure, which means this data will have to eventually be mapped with some API to determine the surronding streets of the event venue so we can determine the impact of said events on traffic. Ultimately, I believe these columns should be eliminated from the dataset since they will only create more trouble and don't contribute much. Instead, I will convert null values to be an empty string for the time being in case we do need them later on.

#### Dataset Cleaning

In [23]:
# Adjust the columsn involving dates and times to be datetime objects 
dataset4raw['Start Date/Time'] = pd.to_datetime(dataset4raw['Start Date/Time'])
dataset4raw['End Date/Time'] = pd.to_datetime(dataset4raw['End Date/Time'])

# Converts the Police Precinct column to numeric, coercing errors to NaN, then fills
# those NaN values with 0 and converts the column to integers
dataset4raw['Police Precinct'] = pd.to_numeric(dataset4raw['Police Precinct'], errors='coerce')
dataset4raw['Police Precinct'] = dataset4raw['Police Precinct'].fillna(0).astype(int)

# Cleans Event Street Side and Street Closure Type to have empty strings instead of null values and converts them to string type
dataset4raw['Event Street Side'] = dataset4raw['Event Street Side'].fillna('').astype(str)
dataset4raw['Street Closure Type'] = dataset4raw['Street Closure Type'].fillna('').astype(str)

# Cleans community board and event name to have empty strings instead of null values and converts them to string type
dataset4raw['Event Name'] = dataset4raw['Event Name'].fillna('').astype(str)

dataset4raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 502426 entries, 0 to 502425
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Event ID             502426 non-null  int64         
 1   Event Name           502426 non-null  str           
 2   Start Date/Time      502426 non-null  datetime64[us]
 3   End Date/Time        502426 non-null  datetime64[us]
 4   Event Borough        502426 non-null  str           
 5   Event Location       502426 non-null  str           
 6   Event Street Side    502426 non-null  str           
 7   Street Closure Type  502426 non-null  str           
 8   Police Precinct      502426 non-null  int64         
dtypes: datetime64[us](2), int64(2), str(5)
memory usage: 34.5 MB


Exports cleaned data to data/processed 

In [24]:
dataset4raw.to_csv("../../data/processed/dataset4_cleaned.csv", index=False)

In [26]:
print(min(dataset4raw['Start Date/Time']), max(dataset4raw['End Date/Time']))


2024-02-01 01:00:00 2024-12-22 18:00:00
